In [1]:
import pandas as pd
print("Hello from VS Code!")

Hello from VS Code!


In [47]:
from pathlib import Path

output_path = Path("../data/sku_master_with_cost.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)

import pandas as pd
import numpy as np

n = 300

# -------------------------
# BASE DATA
# -------------------------
df = pd.DataFrame({
    "sku_id": [f"SKU_{i}" for i in range(n)],
    "origin_country": np.random.choice(
        ["US","Russia","France","Germany","Sweden","Israel","India"], n
    ),
    "unit_cost": np.random.randint(500, 50000, n),
    "annual_demand": np.random.poisson(lam=50, size=n)
})

# -------------------------
# FNS ANALYSIS
# -------------------------
df["FNS_class"] = pd.qcut(
    df["annual_demand"],
    q=3,
    labels=["Slow","Normal","Fast"]
)

# -------------------------
# ABC ANALYSIS (PARETO)
# -------------------------
df = df.sort_values(by="unit_cost", ascending=False)

df["cum_cost"] = df["unit_cost"].cumsum()
total_cost = df["unit_cost"].sum()
df["cum_cost_pct"] = df["cum_cost"] / total_cost

def abc_class(x):
    if x <= 0.7:
        return "A"
    elif x <= 0.9:
        return "B"
    else:
        return "C"

df["ABC_class"] = df["cum_cost_pct"].apply(abc_class)

# -------------------------
# VED (CRITICALITY)
# -------------------------
df["VED_class"] = np.random.choice(
    ["Vital","Essential","Desirable"],
    n,
    p=[0.2,0.5,0.3]
)

# -------------------------
# GEO RISK
# -------------------------
risk_map = {
    "Russia": 0.7,
    "Israel": 0.5,
    "US": 0.2,
    "Germany": 0.2,
    "France": 0.25,
    "Sweden": 0.25,
    "India": 0.15
}

df["risk_score"] = df["origin_country"].map(risk_map)

# -------------------------
# LEAD TIME
# -------------------------
df["base_lead_time_days"] = np.random.randint(5, 60, n)

# -------------------------
# TIME-DISTANCE
# -------------------------
df["distance_km"] = np.random.randint(50, 5000, n)
df["time_distance_index"] = df["distance_km"] / df["distance_km"].max()

# -------------------------
# STOCKING POLICY
# -------------------------
def stocking_policy(row):
    if row["VED_class"] == "Vital":
        return "Push"
    elif row["FNS_class"] == "Fast" and row["time_distance_index"] > 0.5:
        return "Push"
    else:
        return "Pull"

df["stocking_policy"] = df.apply(stocking_policy, axis=1)

# -------------------------
# COST BASELINE
# -------------------------
df["baseline_inventory"] = (
    df["annual_demand"] * df["base_lead_time_days"] / 365
)

df["baseline_inventory_value"] = (
    df["baseline_inventory"] * df["unit_cost"]
)

# -------------------------
# SAVE
# -------------------------
df.to_csv("../data/sku_master_enriched.csv", index=False)

df.head()



,sku_id,origin_country,unit_cost,annual_demand,FNS_class,cum_cost,cum_cost_pct,ABC_class,VED_class,risk_score,base_lead_time_days,distance_km,time_distance_index,stocking_policy,baseline_inventory,baseline_inventory_value
173,SKU_173,Israel,49881,58,Fast,49881,0.006423,A,Essential,0.50,10,1956,0.391670,Pull,1.589041,79262.958904
257,SKU_257,Sweden,49786,38,Slow,99667,0.012833,A,Essential,0.25,38,3414,0.683620,Pull,3.956164,196961.600000
188,SKU_188,Israel,49681,59,Fast,149348,0.019230,A,Essential,0.50,5,224,0.044854,Pull,0.808219,40153.136986
52,SKU_52,US,49681,50,Normal,199029,0.025627,A,Desirable,0.20,57,3355,0.671806,Pull,7.808219,387920.136986
62,SKU_62,Germany,49578,48,Normal,248607,0.032010,A,Vital,0.20,53,670,0.134161,Push,6.969863,345551.868493


This dataset is synthetically generated to simulate MRO supply chain conditions.

It incorporates multi-criteria SKU intelligence including FNS, ABC, VED, geopolitical risk, lead time, and network constraints.

The dataset will be enhanced using real-world anchors (M5 dataset for demand and trade data for risk calibration) in later phases.

In [51]:
df.head()
df.describe()
df["ABC_class"].value_counts()
df["FNS_class"].value_counts()

FNS_class
Normal    108
Slow      100
Fast       92
Name: count, dtype: int64